In [2]:
BASE_URL = "https://www.noon.com/egypt-en"

product_name = "laptop"

In [19]:
from selenium import webdriver

driver = webdriver.Chrome()
driver.get(BASE_URL)

In [20]:
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from time import sleep

search_input = driver.find_element(By.ID, "search-input")
search_input.send_keys(product_name)
search_input.send_keys(Keys.RETURN)

sleep(3)

In [21]:
PRODUCT_ANCHOR_SELECTOR = "[data-qa=\"plp-product-box\"] a"

product_anchors = driver.find_elements(By.CSS_SELECTOR, PRODUCT_ANCHOR_SELECTOR)
product_links = [anchor.get_attribute("href") for anchor in product_anchors][:5]

product_links

['https://www.noon.com/egypt-en/macbook-air-mgn63-with-13-inch-display-m1-chip-8-core-cpu-and-7-core-gpu-8gb-ram-256gb-ssd-macos-english-arabic-space-grey/N41964615A/p/?o=bf3e37196b557c7b',
 'https://www.noon.com/egypt-en/macbook-air-mw123-13-inch-display-apple-m4-chip-10-core-cpu-8-core-gpu-16gb-ram-256gb-ssd-macos-english-keyboard-international-version-midnight/N70154912V/p/?o=c0b70e57abd5dafa',
 'https://www.noon.com/egypt-en/macbook-pro-mde04-14-inch-display-apple-m5-chip-10-core-cpu-and-10-core-gpu-16gb-ram-512gb-ssd-macos-english-keyboard-international-version-english-space-black/N70248386V/p/?o=a323c074d2bd50af',
 'https://www.noon.com/egypt-en/macbook-pro-mx2h3-14-2-inch-display-apple-m4-pro-chip-12-core-cpu-16-core-gpu-24gb-ram-512gb-ssd-macos-english-keyboard-english-space-black/N70125918V/p/?o=aa643117a24c3e6a',
 'https://www.noon.com/egypt-en/microsoft-surface-laptop-go-12-4-10th-gen-ci5-16g-256gb-win-10-pro-platinum-english-silver/N70127133V/p/?o=fea9e5911757e91d']

In [6]:
driver.quit()

In [10]:
from typing import Callable, Any, Optional, TypeVar
from selenium.webdriver.remote.webelement import WebElement
from selenium.webdriver.remote.webdriver import WebDriver
from selenium.common.exceptions import StaleElementReferenceException, NoSuchElementException
from re import search

T = TypeVar("T")


def skip_if_element_not_found(func: Callable[[WebDriver], T]):
    def wrapper(driver: WebDriver) -> Optional[T]:
        try:
            print("trying", func)
            return func(driver)
        except StaleElementReferenceException:
            print("found stale element")
            return None
        except NoSuchElementException:
            print("missing element")
            return None
    return wrapper


@skip_if_element_not_found
def get_title(driver: WebDriver):
    title_element = driver.find_element(By.TAG_NAME, "h1")
    return title_element.text


@skip_if_element_not_found
def get_price(driver: WebDriver):
    price_container: WebElement = driver.find_element(
        By.CSS_SELECTOR, '[data-qa="div-price-now"]'
    )
    price_text = price_container.find_elements(By.XPATH, ".//*")[1].text
    return float(price_text)


@skip_if_element_not_found
def get_rating(driver: WebDriver):
    rating_element = driver.find_element(
        By.CLASS_NAME, "RatingPreviewStarV2-module-scss-module__0_8vQW__text"
    )
    return float(rating_element.text)


@skip_if_element_not_found
def get_review_count(driver: WebDriver):
    review_count_element = driver.find_element(
        By.CLASS_NAME, "NoonRatingsBasedOnTitle-module-scss-module__aAM0SW__basedOnInfoCtrLoader"
    )
    review_count_text = search(r"[\d,]+", review_count_element.text)
    review_count_text = review_count_text.group(0).replace(",", "").strip()
    return int(review_count_text)


@skip_if_element_not_found
def get_image(driver: WebDriver):
    image_element = driver.find_element(
        By.CLASS_NAME, "GalleryV2-module-scss-module__hlK6zG__imageMagnify"
    )
    return image_element.get_attribute("src")


@skip_if_element_not_found
def get_subvendor(driver: WebDriver):
    subvendor_element = driver.find_element(
        By.CLASS_NAME, "PartnerRatingsV2-module-scss-module__1CV-Aa__soldBy"
    )
    return subvendor_element.text

In [7]:
link = product_links[0]
driver = webdriver.Chrome()
driver.get(link)

sleep(4)

get_title(driver), get_price(driver), get_image(driver), get_rating(driver), get_review_count(driver), get_subvendor(driver)


trying <function get_title at 0x72c7558b4f40>
trying <function get_price at 0x72c7558b4540>
trying <function get_image at 0x72c7558b5300>
trying <function get_rating at 0x72c7558b5080>
trying <function get_review_count at 0x72c7558b51c0>
missing element
trying <function get_subvendor at 0x72c7558b5440>


('MacBook Air MGN63 With 13 Inch Display, M1 Chip 8-Core CPU and 7-Core GPU | 8GB RAM | 256GB SSD | macOS | English/Arabic Space Grey',
 34161.2,
 'https://f.nooncdn.com/p/pnsku/N41964615A/45/_/1768825891/2fa999f6-d509-4599-beeb-0d1350dfba22.jpg?width=800',
 4.6,
 None,
 'dtechstore')

In [12]:
from listing import Listing, PriceType

def get_listing_from_link(driver: WebDriver)-> Listing:
    return Listing(
        name=get_title(driver),
        url=driver.current_url,
        image=get_image(driver),
        price_type=PriceType.DISCRETE,
        price=get_price(driver),
        price_range=None,
        vendor="Noon",
        subvendor=get_subvendor(driver),
        rating=get_rating(driver),
        review_count=get_review_count(driver)
    )

In [18]:
from pandas import DataFrame

listings: list[Listing] = []

def process_links(links: list[str]):
    driver = webdriver.Chrome()
    for link in links:
        driver.get(link)
        listing = get_listing_from_link(driver)
        listings.append(listing)
    driver.close()



process_links(product_links)

DataFrame([listing.to_json() for listing in listings])

trying <function get_title at 0x7ea828306480>
trying <function get_image at 0x7ea828305440>
found stale element
trying <function get_price at 0x7ea828306520>
trying <function get_subvendor at 0x7ea828306160>
trying <function get_rating at 0x7ea828306020>
trying <function get_review_count at 0x7ea828305f80>
missing element
trying <function get_title at 0x7ea828306480>
trying <function get_image at 0x7ea828305440>
trying <function get_price at 0x7ea828306520>
trying <function get_subvendor at 0x7ea828306160>
trying <function get_rating at 0x7ea828306020>
trying <function get_review_count at 0x7ea828305f80>
missing element
trying <function get_title at 0x7ea828306480>
trying <function get_image at 0x7ea828305440>
trying <function get_price at 0x7ea828306520>
trying <function get_subvendor at 0x7ea828306160>
trying <function get_rating at 0x7ea828306020>
trying <function get_review_count at 0x7ea828305f80>
missing element
trying <function get_title at 0x7ea828306480>
trying <function get_i

,name,url,image,price_type,price,price_range,vendor,subvendor,rating,review_count
0,"MacBook Air MGN63 With 13 Inch Display, M1 Chi...",https://www.noon.com/egypt-en/macbook-air-mgn6...,NaN,discrete,35150.0,None,Noon,dtechstore,4.6,None
1,MacBook Pro MDE04 | 14 Inch Display | Apple M5...,https://www.noon.com/egypt-en/macbook-pro-mde0...,https://f.nooncdn.com/p/pnsku/N70248386V/45/_/...,discrete,101198.5,None,Noon,dtechstore,4.5,None
2,MacBook Air MW133 | 13 Inch Display | Apple M4...,https://www.noon.com/egypt-en/macbook-air-mw13...,https://f.nooncdn.com/p/pnsku/N70154910V/45/_/...,discrete,70067.0,None,Noon,Original Stores,4.6,None
3,MacBook Air MC7A4 | 15 Inch Display | M4 Chip ...,https://www.noon.com/egypt-en/macbook-air-mc7a...,https://f.nooncdn.com/p/pnsku/N70154961V/45/_/...,discrete,66821.0,None,Noon,Dr.Mobile,4.6,None
4,"Microsoft Surface Laptop GO 12.4"" 10th Gen Ci5...",https://www.noon.com/egypt-en/microsoft-surfac...,https://f.nooncdn.com/p/pnsku/N70127133V/45/_/...,discrete,24444.0,None,Noon,XPRS,3.7,None
